# Đọc và khám phá chatlog CSV

Notebook này dùng Python standard library, không cần cài `pandas`. Mặc định chỉ xem thống kê; chỉ chạy cell preview khi thật sự cần đọc mẫu hội thoại. Không sao chép nội dung gốc sang repo nộp bài.

In [1]:
from collections import Counter
import csv
from pathlib import Path

CSV_PATH = Path('../data/vlearn-pack/chatlog/chat_history_anonymized_for_hackathon.csv')
REQUIRED_COLUMNS = {'conversation_id', 'turn_id', 'role', 'content'}

if not CSV_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy file: {CSV_PATH.resolve()}')

with CSV_PATH.open(encoding='utf-8-sig', newline='') as source:
    reader = csv.DictReader(source)
    if reader.fieldnames is None:
        raise ValueError('CSV không có hàng header.')
    missing = REQUIRED_COLUMNS - set(reader.fieldnames)
    if missing:
        raise ValueError(f'CSV thiếu cột: {sorted(missing)}')
    columns = reader.fieldnames
    rows = list(reader)

print(f'Đã đọc {len(rows):,} dòng và {len(columns)} cột.')

Đã đọc 2,522 dòng và 22 cột.


In [2]:
roles = Counter(row['role'] for row in rows)
conversations = {row['conversation_id'] for row in rows}
turns = {row['turn_id'] for row in rows}
empty_content = sum(not row['content'].strip() for row in rows)

summary = {
    'so_dong': len(rows),
    'so_hoi_thoai': len(conversations),
    'so_turn': len(turns),
    'role': dict(sorted(roles.items())),
    'noi_dung_rong': empty_content,
}
summary

{'so_dong': 2522,
 'so_hoi_thoai': 585,
 'so_turn': 1261,
 'role': {'student': 1261, 'tutor': 1261},
 'noi_dung_rong': 0}

In [3]:
def shorten(value: str, max_length: int = 120) -> str:
    compact = ' '.join(value.split())
    return compact if len(compact) <= max_length else compact[:max_length - 1] + '…'

def preview(limit: int = 5) -> None:
    """Xem ít dòng nhất cần thiết; content luôn được rút gọn."""
    for row in rows[:limit]:
        print(
            f"{row['conversation_id']} | {row['turn_id']} | "
            f"{row['role']}: {shorten(row['content'])}"
        )

preview(5)

C0001 | T0649 | student: (Trang 37, đoạn được chọn: "tóm tắt nội dung chính trong slide này") tóm tắt nội dung chính trong slide này
C0001 | T0649 | tutor: Xin lỗi bạn, tôi không tìm thấy nội dung cụ thể cho slide 37 trong tài liệu hiện có. Bạn có thể cung cấp thêm thông tin…
C0002 | T0959 | tutor: Dựa trên nội dung tại trang 45, có 4 chiến lược chính để tối ưu hóa prompt và ngữ cảnh (context) trong kỹ thuật lập trì…
C0002 | T0959 | student: (Trang 45, đoạn được chọn: "giải thích 4 chiến lược") giải thích 4 chiến lược
C0002 | T0905 | tutor: Chào bạn, hiện tại tôi không tìm thấy tài liệu tổng hợp đầy đủ cho toàn bộ nội dung của Ngày 04 trong slide được cung c…


## Bước tiếp theo

Dùng `rows` để xây rule mining có thể kiểm lại. Khi lưu evidence/golden set vào bài nộp, chỉ lưu mã `conversation_id`/`turn_id` và bản diễn đạt lại hoặc trích đoạn ngắn đã rà soát.

## Xem đầy đủ một hội thoại

Đổi `C0001` thành mã hội thoại bạn muốn kiểm tra. Chỉ dùng để mining trong phạm vi hackathon; không commit output đầy đủ của cell này.

In [11]:
def show_conversation(conversation_id: str) -> None:
    conversation = [row for row in rows if row['conversation_id'] == conversation_id]
    if not conversation:
        print(f'Không tìm thấy hội thoại: {conversation_id}')
        return

    for row in conversation:
        print('=' * 88)
        print(f"Turn: {row['turn_id']} | Role: {row['role']} | Message: {row['message_id']}")
        print(row['content'])

show_conversation('C0006')

Turn: T0058 | Role: student | Message: M2247
(Trang 66, đoạn được chọn: "xem bài tập thực hành lab day 2 chiều nay ở đaau")
xem bài tập thực hành lab day 2 chiều nay ở đaau
Turn: T0058 | Role: tutor | Message: M2014
Chào bạn, hiện tại trong tài liệu slide của ngày học hôm nay không đề cập cụ thể vị trí lưu trữ bài tập thực hành Lab Day 2. Bạn vui lòng kiểm tra trên trang chủ khóa học của hệ thống quản lý học tập (LMS) hoặc thông báo từ giảng viên để biết chính xác nơi tải bài tập nhé.
